# M1M3TS - Fan Coil Unit Quick Analysis

We know some FCUs aren't working as expected.  
Our goal is 0.05 °C RMS from the target, +9ºC in this case.  
  
It would be best if you could make a command-line tool where I specify the time and we get our results—that can then be run in Notebooks. There is a DurationTime class to parse dates on the command line. 

We don't expect FCU with the heater disabled to deliver the correct temperature. 


Associated tickets:
 * [SITCOM-2122 Create script to evaluate the Fan Coil Units health](https://rubinobs.atlassian.net/browse/SITCOM-2122)

## Input Parameters

In [ ]:
# Reference timestamp
timestamp = "2025-05-10T12:00:00"

# Time interval. Units can be "s" for seconds, "m" for minutes, 
# "h" for hours, or "d" for days.
# Negative values indicate a time before the reference timestamp.
delta_t = "-10s"

# The index of the FCU (Flight Control Unit) to be used.
fcu_index = 10

# The temperature set point value for the FCU in degrees Celsius.
set_point = 9 

## Setup Notebook

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import warnings

from argparse import ArgumentTypeError
from matplotlib.lines import Line2D
from astropy import units as u
from astropy.time import Time, TimeDelta
from astropy.time.core import TimeDeltaMissingUnitWarning

from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.ts.xml.tables.m1m3.fcu_table import FCUTable

In [ ]:
# Initialize an EFD client
efd_client = makeEfdClient()

# Set global font size for labels, titles, and ticks
plt.rcParams.update(
    {
        "axes.grid": True,
        "axes.labelsize": 12,
        "axes.titlesize": 14,
        "axes.formatter.useoffset": False,
        "axes.formatter.use_mathtext": False,
        "axes.formatter.limits": (-100, 100),
        "figure.figsize": (11, 6),
        "font.size": 12,
        "grid.color": "#b0b0b0",
        "grid.linestyle": ":",
        "grid.linewidth": 0.5,
        "grid.alpha": 0.75,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
    }
)

## Helper Functions

In [ ]:
def create_url_for_summit_chronograf(t_start, t_end):
    """Create a URL for the Summit Chronograf to visualize FCU data."""
    base_url = "https://summit-lsp.lsst.codes/chronograf"
    dashboard = "sources/1/dashboards/390"
    
    t_start_str = t_start.isot.replace(":", "%3A")
    t_end_str = t_end.isot.replace(":", "%3A")
    
    url = f"{base_url}/{dashboard}?refresh=Paused&lower={t_start_str}Z&upper={t_end_str}Z"
    return url


def get_time_window(timestamp: str, delta_t: str):
    """Given a timestamp and a duration string, return (t_start, t_end) as 
    astropy Time objects."""
    delta_t_seconds = parse_duration(delta_t)
    if delta_t_seconds <= 0:
        t_end = parse_timestamp(timestamp)
        t_start = t_end + delta_t_seconds
    else:
        t_start = parse_timestamp(timestamp)
        t_end = t_start + delta_t_seconds
    return t_start, t_end


# Function copied from lsst-ts/ts_m1m3_utils
def parse_duration(duration: str) -> TimeDelta:
    """Accept string depicting duration.

    Numbers can be suffixed with character, denomination their lengths.

    Length denominators
    -------------------
    D : days (86400 seconds)
    h : hours (3600 seconds)
    m : minutes (60 seconds)
    s : seconds (1 second)

    Examples
    --------
    '1D 1m' = 86460 seconds
    '1h 1m 30s' = 3690 seconds

    Parameters
    ----------
    duration : `str`
        Duration string. Numbers with know suffixed. Non-sufficed number will
        be treated as seconds.

    Returns
    -------
    seconds : float
        Number of seconds in string.
    """
    if not duration:
        raise ValueError("Duration string cannot be empty.")
    
    muls = {"D": 86400, "h": 3600, "m": 60, "s": 1, "u": 0.001, "n": 0.000001}
    ret: float = 0.0
    current: float = 0.0
    duration = duration.strip()
    sign = 1
    fraction = 0
    
    if duration[0] == "-":
        sign = -1
        duration = duration[1:]
    elif duration[0] == "+":
        duration = duration[1:]

    for s in duration.strip():
        if "0" <= s <= "9":
            if fraction > 0:
                current += (0.1**fraction) * int(s)
                fraction += 1
            else:
                current = current * 10 + int(s)
        elif s == ".":
            fraction = 1
        elif s == " ":
            pass
        else:
            try:
                ret += current * muls[s]
                current = 0.0
            except KeyError:
                raise ArgumentTypeError(f"Unknown suffix: {s}")
            
    return TimeDelta(sign * (ret + current) * u.s)


def parse_timestamp(timestamp):
    """Parse the timestamp string into an astropy Time object."""
    return Time(timestamp, format="isot", scale="utc")


def plot_fcu_temperature(df, fcu_index, t_start, t_end):
    """Plot the FCU temperature data from the DataFrame."""
    title = f"FCU{fcu_index} Temperature Data\n From {t_start.iso} to {t_end.iso}"
    fig, ax = plt.subplots(num=1, clear=True)

    ax.plot(df[f"absoluteTemperature{fcu_index}"], label=f"FCU{fcu_index} Temperature", color="blue", linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel("Temperature (deg C)")
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    ax.legend(loc="upper right", fontsize=10)

    fig.autofmt_xdate(rotation=45, ha='right')
    fig.savefig(f"fcu{fcu_index}_temperature_{t_start.iso.replace(':', '-')}_{t_end.iso.replace(':', '-')}.png", dpi=300)
    plt.show()
    
    
def print_temperature_stats(df, fcu_index, set_point):
    col = f"absoluteTemperature{fcu_index}"
    stats = df[col].agg(['min', 'mean', 'median', 'max', 'std'])
    print(
        f"FCU {fcu_index} Temperature Stats:\n"
        f"  Min:       {stats['min']:.3f} deg_C\n"
        f"  Mean:      {stats['mean']:.3f} deg_C\n"
        f"  Median:    {stats['median']:.3f} deg_C\n"
        f"  Max:       {stats['max']:.3f} deg_C\n"
        f"  Std:       {stats['std']:.3f} deg_C\n"
        f"  Set Point: {set_point:.3f} deg_C\n"
        f"  RMS:       {((df[col] - set_point) ** 2).mean() ** 0.5:.3f} deg_C"
    )
    

def query_fcu_data(fcu_index, t_start, t_end):
    """Query the EFD for FCU data within the specified time window."""
    df = getEfdData(
        efd_client, 
        topic="lsst.sal.MTM1M3TS.thermalData",
        columns=["timestamp", f"absoluteTemperature{fcu_index}"],
        begin=t_start,
        end=t_end
    )
    return df


def fcu_quick_analysis(
    fcu_index, 
    timestamp, 
    delta_t, 
    set_point, 
    plot=True, 
    show_url=True
):
    """
    Perform a quick analysis of FCU temperature data. 
    
    Parameters
    ----------
    fcu_index : int
        The FCU index to analyze (1-10).
    timestamp : str
        The timestamp in ISO format (e.g., "2025-05-10T12:00:00").
    delta_t : str
        The duration string (e.g., "-10s" for 10 seconds before the timestamp).
    set_point : float
        The temperature set point for the FCU in degrees Celsius.
    plot : bool, optional
        Whether to plot the temperature data. Default is True.
    show_url : bool, optional
        Whether to show the URL for Summit Chronograf. Default is True.
    """
    warnings.filterwarnings("ignore", category=TimeDeltaMissingUnitWarning)
    
    t_start, t_end = get_time_window(timestamp, delta_t)
    print(f"Time window: {t_start.iso} to {t_end.iso}")

    df = query_fcu_data(fcu_index, t_start, t_end)

    if df.empty:
        print("No data found for the specified time window.")
        return

    print_temperature_stats(df, fcu_index, set_point)
    if show_url:
        url = create_url_for_summit_chronograf(t_start, t_end)
        print(f"View the data in Summit Chronograf:\n  {url}")    
    if plot:
        plot_fcu_temperature(df, fcu_index, t_start, t_end)
    

## Analysis

In [ ]:
fcu_quick_analysis(fcu_index, timestamp, delta_t, set_point, plot=True, show_url=True)